# Stroke Prediction

## Importing necessary libraries

In [1]:
import numpy as np
import pandas as pd 

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
import lightgbm as lgb
from sklearn.model_selection import GridSearchCV
import optuna
from optuna.samplers import TPESampler
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, fbeta_score, make_scorer

## Import dataset

In [2]:
df = pd.read_csv('../dataset/healthcare-dataset-stroke-data.csv')

In [3]:
df.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5110 entries, 0 to 5109
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 5110 non-null   int64  
 1   gender             5110 non-null   object 
 2   age                5110 non-null   float64
 3   hypertension       5110 non-null   int64  
 4   heart_disease      5110 non-null   int64  
 5   ever_married       5110 non-null   object 
 6   work_type          5110 non-null   object 
 7   Residence_type     5110 non-null   object 
 8   avg_glucose_level  5110 non-null   float64
 9   bmi                4909 non-null   float64
 10  smoking_status     5110 non-null   object 
 11  stroke             5110 non-null   int64  
dtypes: float64(3), int64(4), object(5)
memory usage: 479.2+ KB


In [5]:
df.describe()

,id,age,hypertension,heart_disease,avg_glucose_level,bmi,stroke
count,5110.000000,5110.000000,5110.000000,5110.000000,5110.000000,4909.000000,5110.000000
mean,36517.829354,43.226614,0.097456,0.054012,106.147677,28.893237,0.048728
std,21161.721625,22.612647,0.296607,0.226063,45.283560,7.854067,0.215320
min,67.000000,0.080000,0.000000,0.000000,55.120000,10.300000,0.000000
25%,17741.250000,25.000000,0.000000,0.000000,77.245000,23.500000,0.000000
50%,36932.000000,45.000000,0.000000,0.000000,91.885000,28.100000,0.000000
75%,54682.000000,61.000000,0.000000,0.000000,114.090000,33.100000,0.000000
max,72940.000000,82.000000,1.000000,1.000000,271.740000,97.600000,1.000000


In [6]:
df.describe(include=[object])

,gender,ever_married,work_type,Residence_type,smoking_status
count,5110,5110,5110,5110,5110
unique,3,2,5,2,4
top,Female,Yes,Private,Urban,never smoked
freq,2994,3353,2925,2596,1892


## Rename column for consistent formatting of name

In [7]:
df.rename(columns={'Residence_type': 'residence_type'}, inplace=True)

In [8]:
pd.set_option('display.max_columns', None)

In [9]:
pd.set_option('display.max_rows', None)

### Impute missing values in BMI column
For 201 entries missing BMI values, these entries will be imputed with the median BMI value

In [10]:
median_bmi = df['bmi'].median()
df['bmi'].fillna(median_bmi, inplace=True)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_18656\2337012406.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['bmi'].fillna(median_bmi, inplace=True)


In [11]:
print(df['bmi'].isna().sum())

0


## Feature Engineering

### Converting categorical columns to numerical columns

In [12]:
df['male'] = (df['gender'] == 'Male').astype(int)

In [13]:
df['ever_married'] = (df['ever_married'] == 'Yes').astype(int)

In [14]:
df['urban'] = (df['residence_type'] == 'Urban').astype(int)

In [15]:
df['smoke'] = ((df['smoking_status'] == 'formerly smoked') | (df['smoking_status'] == 'smokes')).astype(int)

In [16]:
df['have_work'] = ((df['work_type'] == 'Govt_job') | (df['work_type'] == 'Private') | (df['work_type'] == 'Self-employed')).astype(int)

In [17]:
df['hypertension_or_heart_disease'] = df['hypertension'] | df['heart_disease']

In [18]:
df = pd.get_dummies(df, columns=['work_type', 'smoking_status'], drop_first = False)

### Extra column for the age of married people

In [19]:
df['age_ever_married'] = df['age'] * df['ever_married']

In [20]:
df['age_hypertension'] = df['age'] * df['hypertension']

In [21]:
df['age_heart_disease'] = df['age'] * df['heart_disease']

In [22]:
df['age_bmi'] = df['age'] * df['bmi']

In [23]:
df['age_avg_glucose_level'] = df['age'] * df['avg_glucose_level']

In [24]:
df['bmi_avg_glucose_level'] = df['bmi'] * df['avg_glucose_level']

In [25]:
df['avg_glucose_level_hypertension'] = df['avg_glucose_level'] * df['hypertension']

### Creating columns representing outstanding values for BMI and Average Glucose Level

In [26]:
df['obese'] = df['bmi'] >= 30

In [27]:
df['high_avg_glucose_level'] = df['avg_glucose_level'] >= 150

In [28]:
df['obese_high_avg_glucose_level'] = df['obese'] | df['high_avg_glucose_level']

### Creating column representing high age

In [29]:
df['old'] = df['age'] >= 65

### Converting boolean columns to numerical

In [30]:
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

### Accounting for multiple risks

In [31]:
df['risk_count'] = df['obese'] + df['high_avg_glucose_level'] + df['old'] + df['hypertension'] + df['heart_disease']

### Reformatting column names

In [32]:
df.rename(columns={'work_type_Govt_job': 'work_type_govt_job', 'work_type_Never_worked': 'work_type_never_worked', 'work_type_Private': 'work_type_private', 'work_type_Self-employed': 'work_type_self_employed'}, inplace=True)

In [33]:
df.rename(columns={'smoking_status_Unknown': 'smoking_status_unknown', 'smoking_status_formerly smoked': 'smoking_status_formerly_smoked', 'smoking_status_never smoked': 'smoking_status_never_smoked'}, inplace=True)

In [34]:
df.drop(columns=['gender', 'residence_type'], inplace=True)

### Log Transforming BMI and Average Glucose Level columns

In [35]:
df['log_bmi'] = np.log(df['bmi'])

In [36]:
df['log_avg_glucose_level'] = np.log(df['avg_glucose_level'])

In [37]:
df.head()

,id,age,hypertension,heart_disease,ever_married,avg_glucose_level,bmi,stroke,male,urban,smoke,have_work,hypertension_or_heart_disease,work_type_govt_job,work_type_never_worked,work_type_private,work_type_self_employed,work_type_children,smoking_status_unknown,smoking_status_formerly_smoked,smoking_status_never_smoked,smoking_status_smokes,age_ever_married,age_hypertension,age_heart_disease,age_bmi,age_avg_glucose_level,bmi_avg_glucose_level,avg_glucose_level_hypertension,obese,high_avg_glucose_level,obese_high_avg_glucose_level,old,risk_count,log_bmi,log_avg_glucose_level
0,9046,67.0,0,1,1,228.69,36.6,1,1,1,1,1,1,0,0,1,0,0,0,1,0,0,67.0,0.0,67.0,2452.2,15322.23,8370.054,0.00,1,1,1,1,4,3.600048,5.432367
1,51676,61.0,0,0,1,202.21,28.1,1,0,0,0,1,0,0,0,0,1,0,0,0,1,0,61.0,0.0,0.0,1714.1,12334.81,5682.101,0.00,0,1,1,0,1,3.335770,5.309307
2,31112,80.0,0,1,1,105.92,32.5,1,1,0,0,1,1,0,0,1,0,0,0,0,1,0,80.0,0.0,80.0,2600.0,8473.60,3442.400,0.00,1,0,1,1,3,3.481240,4.662684
3,60182,49.0,0,0,1,171.23,34.4,1,0,1,1,1,0,0,0,1,0,0,0,0,0,1,49.0,0.0,0.0,1685.6,8390.27,5890.312,0.00,1,1,1,0,2,3.538057,5.143008
4,1665,79.0,1,0,1,174.12,24.0,1,0,0,0,1,1,0,0,0,1,0,0,0,1,0,79.0,79.0,0.0,1896.0,13755.48,4178.880,174.12,0,1,1,1,3,3.178054,5.159745


In [38]:
df.shape

(5110, 36)

## Model Building

### Splitting to training, validation, and testing sets

In [39]:
columns_name = columns_name = [col for col in df.columns if col != 'id']

In [40]:
X = df[columns_name]
X.drop('stroke', axis=1, inplace=True)
y = df['stroke']

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_18656\91966498.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X.drop('stroke', axis=1, inplace=True)


In [41]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 42, stratify = y)
X_val, X_test, y_val, y_test = train_test_split(X_test, y_test, test_size = 0.5, random_state = 42, stratify = y_test)

### Create base random forest classifier model

In [42]:
base_classifier = lgb.LGBMClassifier(random_state=42, verbose=-1)

### Create grid for hyperparameter tuning

In [43]:
param_grid_lgbm = {
    'num_leaves': [15, 31, 63, 127],
    'max_depth': [3, 5, 7, 10, -1], 
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'n_estimators': [50, 100, 200, 300],
    'min_child_samples': [5, 10, 20, 30],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'reg_alpha': [0, 0.1, 0.5, 1], 
    'reg_lambda': [0, 0.1, 0.5, 1],  
    'scale_pos_weight': [1, 3, 5, 10, 15, 20]  
}

In [44]:
f2_scorer = make_scorer(fbeta_score, beta=2)

In [45]:
grid_lgbm = GridSearchCV(
    estimator=base_classifier,
    param_grid=param_grid_lgbm,
    scoring=f2_scorer,
    cv=5,
    verbose=2,
    n_jobs=-1
)

### Fitting base classifier on unsampled data

In [46]:
base_classifier.fit(X_train, y_train)

LGBMClassifier(random_state=42, verbose=-1)

In [47]:
importances = base_classifier.feature_importances_

In [48]:
feature_imp_df = pd.DataFrame({'Feature':[col for col in X_train], 'Importance': importances}).sort_values(
    'Importance', ascending=False)

In [49]:
print(feature_imp_df)

                           Feature  Importance
4                avg_glucose_level         463
23                         age_bmi         418
25           bmi_avg_glucose_level         417
5                              bmi         402
24           age_avg_glucose_level         347
0                              age         188
26  avg_glucose_level_hypertension         149
20                age_ever_married         133
21                age_hypertension          77
22               age_heart_disease          56
13               work_type_private          45
7                            urban          43
6                             male          38
18     smoking_status_never_smoked          36
31                      risk_count          36
11              work_type_govt_job          25
19           smoking_status_smokes          24
8                            smoke          22
17  smoking_status_formerly_smoked          20
10   hypertension_or_heart_disease          17
14         wo

In [50]:
# retained_columns = ['avg_glucose_level', 'age_bmi', 'bmi_avg_glucose_level', 'bmi', 'age_avg_glucose_level', 'age', 'avg_glucose_level_hypertension', 'age_ever_married', 'age_hypertension', 'work_type_private']

In [51]:
# retained_columns = ['avg_glucose_level', 'age_bmi', 'bmi_avg_glucose_level', 'bmi', 'age_avg_glucose_level', 'age', 'avg_glucose_level_hypertension', 'age_ever_married']

In [52]:
# retained_columns = ['avg_glucose_level', 'age_bmi', 'bmi_avg_glucose_level', 'bmi', 'age_avg_glucose_level', 'age', 'avg_glucose_level_hypertension', 'age_ever_married', 'age_hypertension']

In [53]:
# retained_columns = ['avg_glucose_level', 'age_bmi', 'bmi_avg_glucose_level', 'bmi', 'age_avg_glucose_level', 'age', 'avg_glucose_level_hypertension', 'age_ever_married', 'age_hypertension', 'age_heart_disease']

In [54]:
# retained_columns = ['avg_glucose_level', 'bmi', 'age', 'age_ever_married', 'age_hypertension', 'age_heart_disease', 'urban', 'male', 'risk_count', 'work_type_self_employed']

In [55]:
# retained_columns = ['avg_glucose_level', 'age', 'bmi', 'age_ever_married', 'urban', 'hypertension', 'male', 'work_type_self_employed', 'smoke']

In [56]:
retained_columns = ['avg_glucose_level', 'age', 'bmi', 'age_ever_married', 'urban', 'hypertension', 'male', 'work_type_self_employed', 'smoke', 'smoking_status_never_smoked']

In [57]:
# retained_columns = ['avg_glucose_level', 'age', 'bmi', 'urban', 'hypertension', 'male', 'work_type_self_employed', 'smoke', 'smoking_status_never_smoked', 'work_type_private']

In [58]:
# retained_columns = ['avg_glucose_level', 'age', 'bmi', 'urban', 'hypertension', 'male', 'work_type_self_employed', 'smoke', 'smoking_status_never_smoked']

In [59]:
# retained_columns = ['avg_glucose_level', 'age', 'bmi', 'urban', 'hypertension', 'male', 'work_type_self_employed', 'smoke', 'smoking_status_never_smoked', 'work_type_private', 'smoking_status_formerly_smoked']

In [60]:
X_train_retained = X_train[retained_columns]

In [61]:
X_val_retained = X_val[retained_columns]

### Create optimizing function for hyperparameter tuning with Optuna

In [62]:
MIN_PRECISION = 0.15

In [63]:
def objective_lgbm(trial):
    params = {
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 80),
        'min_child_weight': trial.suggest_float('min_child_weight', 0.001, 5, log=True), 
        'min_split_gain': trial.suggest_float('min_split_gain', 0, 3), 
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.2, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'subsample_freq': trial.suggest_int('subsample_freq', 1, 7),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.001, 10, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.001, 10, log=True),
        'max_delta_step': trial.suggest_float('max_delta_step', 0, 5),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 5, 25),
        'boosting_type': 'gbdt',  
        'max_bin': trial.suggest_int('max_bin', 200, 400),
        'random_state': 42,
        'verbose': -1,
    }
    
    model = lgb.LGBMClassifier(**params)
    
    model.fit(X_train_retained, y_train)
    
    y_pred_val = model.predict(X_val_retained)
    
    recall = recall_score(y_val, y_pred_val)
    precision = precision_score(y_val, y_pred_val)
    f2_score = fbeta_score(y_val, y_pred_val, beta=2)
    
    # if precision < MIN_PRECISION:
    #     return 0
    # return recall + (precision * 0.01)
    return f2_score + recall * 0.001 + precision * 0.00001

In [64]:
# grid_lgbm.fit(X_train_retained, y_train)

In [65]:
study_lgbm = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
study_lgbm.optimize(objective_lgbm, n_trials=500, show_progress_bar=True)

[I 2026-01-30 15:23:19,676] A new study created in memory with name: no-name-8db0f080-4bdc-4713-ad39-551946fcba6f


  0%|          | 0/500 [00:00<?, ?it/s]

c:\Users\Lenovo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


[I 2026-01-30 15:23:19,999] Trial 0 finished with value: 0.0 and parameters: {'num_leaves': 69, 'max_depth': 15, 'min_child_samples': 61, 'min_child_weight': 0.16383993835282307, 'min_split_gain': 0.46805592132730955, 'learning_rate': 0.008889607461211927, 'n_estimators': 123, 'subsample': 0.9330880728874675, 'subsample_freq': 5, 'colsample_bytree': 0.8540362888980227, 'reg_alpha': 0.0012087541473056963, 'reg_lambda': 7.579479953348009, 'max_delta_step': 4.162213204002109, 'scale_pos_weight': 9.246782213565524, 'max_bin': 236}. Best is trial 0 with value: 0.0.
[I 2026-01-30 15:23:20,140] Trial 1 finished with value: 0.40441559641372143 and parameters: {'num_leaves': 44, 'max_depth': 6, 'min_child_samples': 47, 'min_child_weight': 0.039605150456850806, 'min_split_gain': 0.8736874205941257, 'learning_rate': 0.04777437867054351, 'n_estimators': 155, 'subsample': 0.6460723242676091, 'subsample_freq': 3, 'colsample_bytree': 0.728034992108518, 'reg_alpha': 1.382623217936987, 'reg_lambda': 0.

c:\Users\Lenovo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


...

[I 2026-01-30 15:27:53,533] Trial 372 finished with value: 0.0 and parameters: {'num_leaves': 105, 'max_depth': 7, 'min_child_samples': 35, 'min_child_weight': 0.002346723905040747, 'min_split_gain': 2.057286730207819, 'learning_rate': 0.006688500205116276, 'n_estimators': 473, 'subsample': 0.7738331532636663, 'subsample_freq': 5, 'colsample_bytree': 0.8541644711985152, 'reg_alpha': 0.07890474445268787, 'reg_lambda': 2.1394342716386046, 'max_delta_step': 0.3921474222792014, 'scale_pos_weight': 15.748085862705604, 'max_bin': 386}. Best is trial 365 with value: 0.46757692052242156.
[I 2026-01-30 15:27:54,147] Trial 373 finished with value: 0.4450950603062078 and parameters: {'num_leaves': 109, 'max_depth': 7, 'min_child_samples': 39, 'min_child_weight': 0.0017454170083910656, 'min_split_gain': 1.9819917887545762, 'learning_rate': 0.007235359292919989, 'n_estimators': 490, 'subsample': 0.7679852116849728, 'subsample_freq': 5, 'colsample_bytree': 0.8259837152929214, 'reg_alpha': 0.08868435

c:\Users\Lenovo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


[I 2026-01-30 15:28:01,366] Trial 386 finished with value: 0.0 and parameters: {'num_leaves': 106, 'max_depth': 7, 'min_child_samples': 37, 'min_child_weight': 0.0015882260301271673, 'min_split_gain': 2.222186121433633, 'learning_rate': 0.006687909612649783, 'n_estimators': 478, 'subsample': 0.7999633418102711, 'subsample_freq': 5, 'colsample_bytree': 0.8165979874015129, 'reg_alpha': 0.04823110493771191, 'reg_lambda': 1.1443881238284033, 'max_delta_step': 0.9174555158854161, 'scale_pos_weight': 14.731777817963419, 'max_bin': 391}. Best is trial 365 with value: 0.46757692052242156.
[I 2026-01-30 15:28:02,076] Trial 387 finished with value: 0.45348088860994523 and parameters: {'num_leaves': 113, 'max_depth': 8, 'min_child_samples': 39, 'min_child_weight': 0.0011287083461342295, 'min_split_gain': 2.0467535828877725, 'learning_rate': 0.0076165677827326715, 'n_estimators': 488, 'subsample': 0.7633658533806783, 'subsample_freq': 5, 'colsample_bytree': 0.8327459576960107, 'reg_alpha': 0.08096

c:\Users\Lenovo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


[I 2026-01-30 15:28:45,361] Trial 450 finished with value: 0.0 and parameters: {'num_leaves': 99, 'max_depth': 8, 'min_child_samples': 33, 'min_child_weight': 0.0020577833289426844, 'min_split_gain': 1.9862724475585363, 'learning_rate': 0.005871809563174264, 'n_estimators': 145, 'subsample': 0.847427995671451, 'subsample_freq': 5, 'colsample_bytree': 0.8567896965847186, 'reg_alpha': 0.06130788406500791, 'reg_lambda': 2.563516193521104, 'max_delta_step': 3.0441051021478356, 'scale_pos_weight': 16.85903254906113, 'max_bin': 386}. Best is trial 445 with value: 0.4694008708708709.
[I 2026-01-30 15:28:46,002] Trial 451 finished with value: 0.4253070012870013 and parameters: {'num_leaves': 92, 'max_depth': 7, 'min_child_samples': 29, 'min_child_weight': 0.0021994266546946336, 'min_split_gain': 1.9271190293311218, 'learning_rate': 0.006531520987790513, 'n_estimators': 477, 'subsample': 0.7680333839254299, 'subsample_freq': 5, 'colsample_bytree': 0.5930645991216087, 'reg_alpha': 0.109595180553

In [66]:
# best_params_lgbm = grid_lgbm.best_params_

In [67]:
best_params_lgbm = study_lgbm.best_params

In [68]:
print(best_params_lgbm)

{'num_leaves': 118, 'max_depth': 8, 'min_child_samples': 34, 'min_child_weight': 0.0019900294648756926, 'min_split_gain': 2.0411153790841694, 'learning_rate': 0.006957001076177289, 'n_estimators': 495, 'subsample': 0.7708065302692244, 'subsample_freq': 5, 'colsample_bytree': 0.7646621089769187, 'reg_alpha': 0.11567144028375737, 'reg_lambda': 2.91568675362154, 'max_delta_step': 1.7246538927085886, 'scale_pos_weight': 13.54579054162353, 'max_bin': 373}


In [69]:
final_classifier = lgb.LGBMClassifier(**best_params_lgbm, random_state=42, verbose=-1)

In [70]:
final_classifier.fit(X_train_retained, y_train)

LGBMClassifier(colsample_bytree=0.7646621089769187,
               learning_rate=0.006957001076177289, max_bin=373,
               max_delta_step=1.7246538927085886, max_depth=8,
               min_child_samples=34, min_child_weight=0.0019900294648756926,
               min_split_gain=2.0411153790841694, n_estimators=495,
               num_leaves=118, random_state=42, reg_alpha=0.11567144028375737,
               reg_lambda=2.91568675362154, scale_pos_weight=13.54579054162353,
               subsample=0.7708065302692244, subsample_freq=5, verbose=-1)

In [71]:
y_pred_val = final_classifier.predict(X_val_retained)

In [72]:
y_proba_val = final_classifier.predict_proba(X_val_retained)[:, 1]

In [73]:
print(confusion_matrix(y_val, y_pred_val))

[[647  82]
 [ 13  24]]


In [74]:
print(accuracy_score(y_val, y_pred_val))

0.8759791122715405


In [75]:
print(precision_score(y_val, y_pred_val))

0.22641509433962265


In [76]:
print(recall_score(y_val, y_pred_val))

0.6486486486486487


In [77]:
print(fbeta_score(y_val, y_pred_val, beta=2))

0.47244094488188976


### Threshold tuning

In [78]:
best_threshold = 0.5

In [79]:
best_precision_score = 0

In [80]:
best_recall_score = 0

In [81]:
best_f2_score = 0

In [82]:
for threshold in np.arange(0.1, 0.95, 0.05):
    y_pred_val_threshold = (y_proba_val >= threshold).astype(int)
    recall = recall_score(y_val, y_pred_val_threshold)
    precision = precision_score(y_val, y_pred_val_threshold)
    f2_score = fbeta_score(y_val, y_pred_val_threshold, beta=2)
    
    print(f'Threshold {threshold} has F2 score {f2_score}')
    
    if precision >= 0.15:
        current_recall_score = recall
        if current_recall_score > best_recall_score:
            best_recall_score = current_recall_score
            best_precision_score = precision
            best_threshold = threshold
        elif current_recall_score == best_recall_score and precision > best_precision_score:
            best_precision_score = precision
            best_threshold = threshold
        
        # current_f2_score = f2_score
        # if current_f2_score > best_f2_score:
        #     best_f2_score = current_f2_score
        #     best_threshold = threshold 

Threshold 0.1 has F2 score 0.34232365145228216
Threshold 0.15000000000000002 has F2 score 0.3611738148984199
Threshold 0.20000000000000004 has F2 score 0.3864734299516908
Threshold 0.25000000000000006 has F2 score 0.3875968992248062
Threshold 0.30000000000000004 has F2 score 0.3977272727272727
Threshold 0.3500000000000001 has F2 score 0.38922155688622756
Threshold 0.40000000000000013 has F2 score 0.4139072847682119
Threshold 0.45000000000000007 has F2 score 0.43956043956043955
Threshold 0.5000000000000001 has F2 score 0.47244094488188976
Threshold 0.5500000000000002 has F2 score 0.3879310344827586
Threshold 0.6000000000000002 has F2 score 0.35714285714285715
Threshold 0.6500000000000001 has F2 score 0.31088082901554404
Threshold 0.7000000000000002 has F2 score 0.2023121387283237
Threshold 0.7500000000000002 has F2 score 0.09554140127388536
Threshold 0.8000000000000002 has F2 score 0.0
Threshold 0.8500000000000002 has F2 score 0.0
Threshold 0.9000000000000002 has F2 score 0.0


c:\Users\Lenovo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Lenovo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [83]:
print(best_threshold)

0.40000000000000013


In [84]:
print(best_recall_score)

0.6756756756756757


### Retrain on training + validation set

In [85]:
X_train_full = pd.concat([X_train_retained, X_val_retained], axis=0, ignore_index=True)
y_train_full = pd.concat([y_train, y_val], axis=0, ignore_index=True)

In [86]:
final_classifier.fit(X_train_full, y_train_full)

LGBMClassifier(colsample_bytree=0.7646621089769187,
               learning_rate=0.006957001076177289, max_bin=373,
               max_delta_step=1.7246538927085886, max_depth=8,
               min_child_samples=34, min_child_weight=0.0019900294648756926,
               min_split_gain=2.0411153790841694, n_estimators=495,
               num_leaves=118, random_state=42, reg_alpha=0.11567144028375737,
               reg_lambda=2.91568675362154, scale_pos_weight=13.54579054162353,
               subsample=0.7708065302692244, subsample_freq=5, verbose=-1)

### Evaluate on test set

In [87]:
X_test_retained = X_test[retained_columns]

In [88]:
y_pred_test = final_classifier.predict(X_test_retained)

In [89]:
y_proba_test = final_classifier.predict_proba(X_test_retained)[:, 1]

In [90]:
print(confusion_matrix(y_test, y_pred_test))

[[633  96]
 [ 11  27]]


In [91]:
print(accuracy_score(y_test, y_pred_test))

0.8604954367666232


In [92]:
print(precision_score(y_test, y_pred_test))

0.21951219512195122


In [93]:
print(recall_score(y_test, y_pred_test))

0.7105263157894737


In [94]:
print(fbeta_score(y_test, y_pred_test, beta=2))

0.4909090909090909


### Test with best threshold 0.55

In [95]:
y_pred_test_threshold = (y_proba_test >= best_threshold).astype(int)

In [96]:
print(confusion_matrix(y_test, y_pred_test_threshold))

[[597 132]
 [  9  29]]


In [97]:
print(accuracy_score(y_test, y_pred_test_threshold))

0.8161668839634941


In [98]:
print(precision_score(y_test, y_pred_test_threshold))

0.18012422360248448


In [99]:
print(recall_score(y_test, y_pred_test_threshold))

0.7631578947368421


In [100]:
print(fbeta_score(y_test, y_pred_test_threshold, beta=2))

0.46325878594249204


In [101]:
importances_final = final_classifier.feature_importances_

In [102]:
feature_imp_df_final = pd.DataFrame({'Feature':[col for col in X_train[retained_columns]], 'Importance': importances_final}).sort_values(
    'Importance', ascending=False)

In [103]:
print(feature_imp_df_final)

                       Feature  Importance
2                          bmi        3414
0            avg_glucose_level        2735
1                          age        2363
3             age_ever_married        1146
5                 hypertension         147
8                        smoke         108
9  smoking_status_never_smoked          91
7      work_type_self_employed          79
6                         male          17
4                        urban          10
